# Урок 02. Когда ML действительно лучше простого метода?

**G-Analytics Business Pulse — практикум по честному эксперименту.**

Цель — сравнить 3 модели по одному и тому же будущему периоду, не подглядывая в будущие данные. Это **синтетическая демонстрация**, а не результат на живом бизнесе. Подробности в `02_model_comparison.md`.

## 1. Подготовка

Работать можно из корня проекта или папки `lessons`. Код ниже определяет корень автоматически.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
if not (root / "src").exists():
    raise RuntimeError("Открой notebook из корня проекта или папки lessons")
sys.path.insert(0, str(root))
from src.data import load_daily
from scripts.compare_models import METHODS, compare
import pandas as pd

daily, source = load_daily(root / "data/demo/daily_sales.csv")
print(source)
print(daily.head())

## 2. Проверяем определение задачи

Прогнозируем выручку на **следующий день**. Можем использовать календарь и выручку за прошлые дни. Фактическое число заказов *завтра* неизвестно — использовать его нельзя.

**Проверь себя:** какие из твоих предложенных признаков доступны до наступления дня?

In [ ]:
from src.model import FEATURES
print("Признаки, используемые ML:")
print(FEATURES)
assert "orders" not in FEATURES
assert "aov" not in FEATURES

## 3. Три конкурирующих подхода

- **Naive:** продажи 7 дней назад.
- **Weekday median:** медиана аналогичных дней за последние 4 недели.
- **Gradient Boosting:** обучаемая модель, использующая календарные и исторические признаки.

Делим данные на Train, Validation и Test **по времени**. Выбираем метод по Validation. Test используем только для оценки выбранного решения.

In [ ]:
validation, test, report = compare(daily)
rows=[]
for method in METHODS:
    rows.append({
        "method": method,
        "validation_WAPE_%": report["validation_metrics"][method]["wape_pct"],
        "test_WAPE_%": report["test_metrics"][method]["wape_pct"],
        "test_MAE": report["test_metrics"][method]["mae"],
    })
results = pd.DataFrame(rows).sort_values("validation_WAPE_%")
print("Выбран по Validation:", report["selected_method"])
print("Периоды:", report["training_dates"], report["validation_dates"], report["test_dates"])
display(results)

## 4. Смотрим прогнозы на тех же днях

Метрики не заменяют визуальный анализ ошибок. Проверь, в какие даты модель сильно промахивается и не совпадают ли они с искусственно внедрёнными инцидентами. Не объявляй любое отклонение реальным ущербом.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(13, 5))
plt.plot(test["date"], test["revenue"], label="Факт")
for method in METHODS:
    plt.plot(test["date"], test[method], label=method, alpha=0.75)
plt.xlabel("Дата")
plt.ylabel("Выручка (условные ед.)")
plt.title("One-day-ahead backtest: одинаковый период для всех методов")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 5. Диагностика ошибок выбранного метода

Посмотри дни с наибольшей ошибкой и скажи, можешь ли ты **по имеющимся данным** установить причину. Это отдельный навык: не путать гипотезу с доказательством.

In [ ]:
display(test.nlargest(8, "absolute_error")[["date", "revenue", "chosen_forecast", "error", "absolute_error", "orders"]])

## 6. Твоя инженерная экспертиза

1. Почему мы выбираем метод **по валидации**, хотя на тесте у ML WAPE немного меньше?
2. Можно ли использовать фактический средний чек прогнозируемого дня? Почему?
3. Если модель показывает WAPE 20%, означает ли это, что 20% продаж потеряно?
4. Какие уведомления должны приходить срочно, а какие можно включать в регулярную сводку?

**Только после объяснения** можешь изменять код. ИИ может написать его, но ты отвечаешь за корректность эксперимента.